# Fake News Detection with ModernBERT  
**Objective**: Classify news articles into human-written vs. LLM-generated fake/real content using a two-stage BERT model.

# 0. Setup & Configuration

## 0.1 Core Libraries & Drive Mounting  
Import critical libraries (`pandas`, `transformers`, etc.) and mount Google Drive for dataset access.  
*Why?* Ensures environment setup matches project requirements.

In [ ]:
import re
import nltk
import os
import json
import pandas as pd
import torch
from sklearn.utils import resample
from transformers import BertTokenizer, BertModel
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


# 1. Data Acquisition & Initial Exploration

### FakeNewsNet Dataset
FAKENEWSNET is an ongoing data collection project for fake news research. It consists of headlines and body texts of fake news articles based on BuzzFeed and PolitiFact. It also collects information about the social engagements of these articles from Twitter.

https://github.com/JinyanSu1/Fakenews-dataset

The orginial fake news and real news are used as HF and HR.
MF and MR are datasets created by adopting the SMP prompting technique via ChatGPT. Generated 97 and 4,084 LLM-generated fake news for PolitiFact and GossipCop, respectively.

The dataset was created based on politifact and gossipcop dataset from FakeNewsNet (https://github.com/KaiDMML/FakeNewsNet/tree/dependabot/pip/certifi-2022.12.7), where the news articles that don't contain title or description has already been excluded.

## 1.1 Loading the FakeNewsNet Dataset  
- Loads human-written and GPT-generated fake/real news from GossipCop and PolitiFact  
- Combines into unified DataFrame with labels: `human_real`, `human_fake`, `gpt_real`, `gpt_fake`  
*Key Insight*: Note the initial class imbalance (8,362 human_real vs 4,181 gpt_fake).

In [ ]:
# Function to read a JSON file
def load_json_file(file_path):
    """Load a JSON file"""
    with open(file_path, "r", encoding="utf-8") as file:
        return json.load(file)

In [ ]:
# Function to process FakeNewsNet data
def process_fakenewsnet_data(file_paths):
    """Read multiple JSON files and organize them into a DataFrame"""
    data_list = []
    label_map = {"HF": "human_fake", "HR": "human_real", "MF": "gpt_fake", "MR": "gpt_real"}

    for category, file_path in file_paths.items():
        data = load_json_file(file_path)
        source = "gossipcop" if "gossipcop" in category else "politifact"
        label = label_map.get(category.split("_")[-1], "unknown")

        for news_id, news_content in data.items():
            if 'text' in news_content:
                data_list.append({
                    "id": news_id,
                    "text": news_content["text"],
                    "title": news_content.get("title", ""),
                    "source": source,
                    "label": label,
                })

    return pd.DataFrame(data_list)

In [ ]:
# Define dataset paths
fake_news_paths = {category: f"/content/drive/MyDrive/ColabNotebooks/W266/FinalProject/FakeNewsNet/{category}.json"
                    for category in ["gossipcop_HF", "gossipcop_HR", "politifact_HF", "politifact_HR",
                                     "gossipcop_MF", "gossipcop_MR", "politifact_MF", "politifact_MR"]}

# Load and process the data
FakeNewsNet_df = process_fakenewsnet_data(fake_news_paths)

# Display the first 5 rows
FakeNewsNet_df.head()

,id,text,title,source,label
0,0,✕ Close Meghan Markle and Prince Harry have an...,As it happened: Prince Harry and Meghan Markle...,gossipcop,human_fake
1,1,Kim Kardashian and Kanye West are pulling out ...,Kim & Kanye Install At-Home Panic Room After P...,gossipcop,human_fake
2,2,Prince Harry and Meghan currently live at Kens...,£1.4million spent renovating Prince Harry and ...,gossipcop,human_fake
3,3,They can't get enough of the Biebs on this sho...,Photos from Dancing With the Stars: Special Gu...,gossipcop,human_fake
4,4,Ben Affleck is keeping life with his three kid...,Jennifer Garner ‘Doesn’t Want’ Her Kids Around...,gossipcop,human_fake


## 1.2 Dataset Health Check  
- Confirms no missing values  
- Displays label distribution:  
  - Human-Real: 8,362  
  - GPT-Fake: 4,181  
  - Human-Fake: 4,181  
- Word count analysis shows GPT-generated text is shorter (mean 225 words) vs human text (587 words)

In [ ]:
# check if there is any missing value
FakeNewsNet_df.isnull().sum()

,0
id,0
text,0
title,0
source,0
label,0


In [ ]:
# check each label category
FakeNewsNet_df['label'].value_counts()

,count
label,
human_real,8362
gpt_real,4301
human_fake,4181
gpt_fake,4181


In [ ]:
# calcalate word count distributon of news text
FakeNewsNet_df["word_count"] = FakeNewsNet_df["text"].apply(lambda x: len(x.split()))
FakeNewsNet_df.groupby("label")["word_count"].describe()

,count,mean,std,min,25%,50%,75%,max
label,,,,,,,,
gpt_fake,4181.0,225.099976,51.704706,8.0,192.0,222.0,255.0,520.0
gpt_real,4301.0,382.680539,344.296195,1.0,263.0,329.0,413.0,5703.0
human_fake,4181.0,465.655106,712.308432,5.0,181.0,337.0,482.0,16373.0
human_real,8362.0,587.115403,1122.563920,4.0,210.0,351.5,578.0,18010.0


 # 2. Data Preparation

## 2.1 Addressing Class Imbalance  
- Undersamples majority class (human_real) to 4,181 samples  
- Creates balanced dataset with equal class representation

In [ ]:
# Get counts for all classes
counts = FakeNewsNet_df['label'].value_counts()
n_samples = min(counts)  # Or set manually (e.g., 4000)

balanced_dfs = []
for label in ["human_real", "human_fake", "gpt_fake", "gpt_real"]:
    df_class = FakeNewsNet_df[FakeNewsNet_df['label'] == label]
    balanced_dfs.append(df_class.sample(n=n_samples, random_state=42))

balanced_FakeNewsNet_df = pd.concat(balanced_dfs)

In [ ]:
!pip install nlpaug nltk

## 2.2 Stratified Dataset Splitting  
- 80% train / 10% val / 10% test split  
- Preserves class distribution in all splits  
*Rationale*: Prevents leakage and ensures representative evaluation.

In [ ]:
from sklearn.model_selection import train_test_split

# Split all 4 classes
def split_dataset(df):
    train, temp = train_test_split(df, test_size=0.2, random_state=42, stratify=df['label'])
    val, test = train_test_split(temp, test_size=0.5, random_state=42, stratify=temp['label'])
    return train, val, test

# Apply to each class
datasets = {}
for label in ["human_real", "human_fake", "gpt_fake", "gpt_real"]:
   df_label = balanced_FakeNewsNet_df[balanced_FakeNewsNet_df['label'] == label]
   datasets[label] = split_dataset(df_label)

# Combine across classes
train_df = pd.concat([d[0] for d in datasets.values()])  # Access train DataFrame using index 0
val_df = pd.concat([d[1] for d in datasets.values()])  # Access val DataFrame using index 1
test_df = pd.concat([d[2] for d in datasets.values()])  # Access test DataFrame using index 2

# Print label distributions
print("Training set label distribution:")
print(train_df['label'].value_counts())
print("\nValidation set label distribution:")
print(val_df['label'].value_counts())
print("\nTest set label distribution:")
print(test_df['label'].value_counts())

# Print dataset sizes
print(f"Training: {len(train_df)}, Validation: {len(val_df)}, Test: {len(test_df)}")

Training set label distribution:
label
human_real    3344
human_fake    3344
gpt_fake      3344
gpt_real      3344
Name: count, dtype: int64

Validation set label distribution:
label
human_real    418
human_fake    418
gpt_fake      418
gpt_real      418
Name: count, dtype: int64

Test set label distribution:
label
human_real    419
human_fake    419
gpt_fake      419
gpt_real      419
Name: count, dtype: int64
Training: 13376, Validation: 1672, Test: 1676


## 2.3 Augmenting Human-Written Text  
- Applies synonym replacement and word swapping to human-generated text  
- Doubles training samples for `human_real` and `human_fake` classes  
*Why?* GPT-generated text often has lower variance - augmentation helps model generalize to human writing styles.

In [ ]:
# To address the poor performance of human-generated data - Data Augmentation before splitting

import nlpaug.augmenter.word as naw
import random
import nltk

# Download the necessary NLTK data package
nltk.download('averaged_perceptron_tagger')
nltk.download('wordnet') # Add to download wordnet data
nltk.download('averaged_perceptron_tagger_eng') # add to download missing package

# Initialize augmenters
synonym_aug = naw.SynonymAug(aug_src='wordnet')  # WordNet-based synonyms
swap_aug = naw.RandomWordAug(action="swap")      # Word swapping

def augment_text(text):
    # Randomly choose an augmentation method
    aug_method = random.choice([synonym_aug, swap_aug])
    return aug_method.augment(text)[0]

# Apply augmentation to human-generated data only
human_train_df = train_df[train_df['label'].isin(['human_real', 'human_fake'])].copy()
augmented_texts = human_train_df['text'].apply(augment_text)

# Add augmented data to training set
augmented_df = pd.DataFrame({
    'text': augmented_texts,
    'label': human_train_df['label'],
    'source': human_train_df['source']
})

# Shuffling the combined data after augmentation
train_df = pd.concat([train_df, augmented_df]).sample(frac=1, random_state=42)

# Ensure no augmented data in val/test sets
assert len(set(train_df.index).intersection(set(val_df.index))) == 0
assert len(set(train_df.index).intersection(set(test_df.index))) == 0

[nltk_data] Downloading package averaged_perceptron_tagger to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger is already up-to-
[nltk_data]       date!
[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package averaged_perceptron_tagger_eng to
[nltk_data]     /root/nltk_data...
[nltk_data]   Package averaged_perceptron_tagger_eng is already up-to-
[nltk_data]       date!


# 3. Text Preprocessing & Feature Engineering

## 3.1 Text Normalization Pipeline

1. Remove HTML tags and URLs  
2. Strip special characters/punctuation  
3. Convert to lowercase  
4. Compact whitespace  
*Example*: "BREAKING: Trump says COVID is HOAX!! (http://..." → "breaking trump says covid is hoax"

In [ ]:
import string

# Remove unwanted characters, links, and HTML tags
def clean_text(text):
    # Remove HTML tags
    text = re.sub(r'<.*?>', '', text)
    # Remove URLs
    text = re.sub(r'http\S+|www\S+|https\S+', '', text)
    # Remove special characters and numbers, keeping only letters
    text = re.sub(r'[^a-zA-Z\s]', '', text)
    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()
    # Convert to lowercase
    text = text.lower()
    return text

# Apply cleaning to datasets
train_df['cleaned_text'] = train_df['text'].apply(clean_text)
val_df['cleaned_text'] = val_df['text'].apply(clean_text)
test_df['cleaned_text'] = test_df['text'].apply(clean_text)

# Print some cleaned samples
print(train_df['cleaned_text'].head())


2138     hit show nbc friends is getting an unauthorize...
6063     dr luke has denied a claim made by kesha that ...
14384    gwen stefani and blake shelton have been toget...
3432     based in a clinic just a stones throw from ken...
4251     james corden has learned to cower every time k...
Name: cleaned_text, dtype: object


In [ ]:
!pip install textstat

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 105.3/105.3 kB 6.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 939.4/939.4 kB 42.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 84.4 MB/s eta 0:00:00


In [ ]:
!pip install vaderSentiment  # If not already installed
nltk.download('punkt')       # For word_tokenize

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 126.0/126.0 kB 9.4 MB/s eta 0:00:00


[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt.zip.


True

## 3.2 Stylometric Feature Extraction  
Computes 10 linguistic features per text:  
- Readability (Flesch, Dale-Chall)  
- Sentiment polarity (VADER)  
- Structural features (! count, avg word length)  
- Complexity metrics (syllables/word)  
*Insight*: Human writing shows higher variance in these features compared to GPT outputs.

In [ ]:
# To address the poor performance of human-generated data - Add linguistic features to help model distinguish human text

import textstat
import string
from nltk.sentiment import SentimentIntensityAnalyzer
from nltk.tokenize import word_tokenize
import nltk
import numpy as np

# Download all required NLTK datasets
nltk.download('punkt')          # For word_tokenize
nltk.download('vader_lexicon')  # For sentiment analysis
nltk.download('punkt_tab')    # Download the missing punkt_tab resource

# Initialize sentiment analyzer
sia = SentimentIntensityAnalyzer()

# Enhanced Feature Extraction
def add_linguistic_features(text):

    # Handle empty text case
    if not text.strip():
        return [0.0]*10  # Return zero features

    features = [
        textstat.flesch_reading_ease(text),
        sia.polarity_scores(text)['compound'],
        len([token for token in word_tokenize(text) if token not in string.printable]),
        textstat.dale_chall_readability_score(text),
        textstat.lexicon_count(text),
        len(re.findall(r'\b[A-Z]+\b', text)),
        text.count('!'),
        len(text.split()),
        np.mean([len(word) for word in text.split()]) if len(text.split()) > 0 else 0.0,
        textstat.syllable_count(text),      # Syllables
    ]

    # Convert to numpy array for NaN handling
    features = np.array(features)
    features = np.nan_to_num(features, nan=0.0, posinf=0.0, neginf=0.0)
    return features.tolist()  # Convert back to list

# Add features to DataFrame
train_df['features'] = train_df['cleaned_text'].apply(add_linguistic_features)
val_df['features'] = val_df['cleaned_text'].apply(add_linguistic_features)
test_df['features'] = test_df['cleaned_text'].apply(add_linguistic_features)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package vader_lexicon to /root/nltk_data...
[nltk_data]   Package vader_lexicon is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


# 4. Model Architecture & Training

## 4.1 ModernBERT Tokenization  
- Uses ModernBERT's 52,000-token vocabulary  
- Truncates/pads to 520 tokens (95th percentile length)  
- Creates `input_ids` and `attention_mask` tensors

In [ ]:
# Use ModernBERT's tokenizer to convert text into tokens and pad/truncate sequences to a fixed length of 520 tokens.
from transformers import AutoTokenizer

# Load ModernBERT tokenizer
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

def tokenize_and_pad(texts, max_length=520):
    # Tokenize text using ModernBERT's tokenizer
    encoding = tokenizer(
        texts,
        truncation=True,  # Truncate texts longer than max_length
        padding="max_length",  # Pad shorter texts to max_length
        max_length=max_length,  # Set the maximum sequence length
        return_tensors="pt"  # Return PyTorch tensor format
    )
    return encoding

# Tokenize and pad datasets
train_encodings = tokenize_and_pad(train_df['cleaned_text'].tolist())
val_encodings = tokenize_and_pad(val_df['cleaned_text'].tolist())
test_encodings = tokenize_and_pad(test_df['cleaned_text'].tolist())

# Check the shape of tokenized data
print(train_encodings['input_ids'].shape)  # Output shape of input_ids


tokenizer_config.json:   0%|          | 0.00/20.8k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/2.13M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/694 [00:00<?, ?B/s]

torch.Size([20064, 520])


## 4.2 Hybrid Dataset Structure  
Combines:  
- Tokenized text sequences  
- Linguistic feature vectors  
- Class labels  
Enables multi-modal learning.

In [ ]:
# Label Encoding: the model requires numerical labels

label_map = {
    "human_real": 0,
    "human_fake": 1,
    "gpt_fake": 2,
    "gpt_real": 3  # Added
}

train_labels = train_df['label'].map(label_map).tolist()
val_labels = val_df['label'].map(label_map).tolist()
test_labels = test_df['label'].map(label_map).tolist()

# Print a few mapped labels
print(train_labels[:10])  # Output first 10 labels


[1, 0, 2, 1, 0, 0, 2, 1, 2, 1]


## 4.3 ModernBERT Customization  
- Loads `answerdotai/ModernBERT-base` pretrained model  
- Replaces classifier head with:  
  `Linear(768+10 features → 4 classes)`  
- Implements Focal Loss (γ=2) to handle residual class imbalance

In [ ]:
# Implement focal loss to handle remaining class imbalanace
from torch.nn import CrossEntropyLoss

class FocalLoss(CrossEntropyLoss):
    def __init__(self, alpha=0.25, gamma=2, **kwargs):
        super().__init__(**kwargs)
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, input, target):
        ce_loss = super().forward(input, target)
        pt = torch.exp(-ce_loss)
        return self.alpha * (1-pt)**self.gamma * ce_loss

In [ ]:
# Implement Class Weighted Loss - to address the underperforming class - human_real and human_fake
import torch
import torch.nn as nn
from transformers import AutoModelForSequenceClassification
from sklearn.utils.class_weight import compute_class_weight

# Get the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

#Dynamic class weighting
class_weights = compute_class_weight(
    'balanced',
    classes=np.unique(train_labels),
    y=train_labels
)
class_weights = torch.tensor(class_weights, dtype=torch.float32).to(device)

model_name = "answerdotai/ModernBERT-base" # Define model_name here
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)
model.classifier = nn.Linear(model.config.hidden_size, 4)

model.loss_fct = FocalLoss()

config.json:   0%|          | 0.00/1.19k [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/599M [00:00<?, ?B/s]

Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


## 4.4 Training Configuration  
- Create PyTorch Dataset and DataLoader
- Load ModernBERT
- Modify Compute Metrics for per-class metric

In [ ]:
!pip install pytorch-ignite

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 343.2/343.2 kB 23.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 3.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 13.8/13.8 MB 112.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 24.6/24.6 MB 87.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 883.7/883.7 kB 45.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 42.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 19.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 4.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 99.1 MB/s eta 0:00:00
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.5.82
    Uninsta

In [ ]:
from ignite.engine import Events, Engine
from ignite.handlers import EarlyStopping
from ignite.metrics import Accuracy
from transformers import Trainer, TrainingArguments # Import TrainingArguments and Trainer
import torch
import numpy as np
from sklearn.metrics import accuracy_score, precision_recall_fscore_support

In [ ]:
 # Create PyTorch Dataset and DataLoader
class FakeNewsDataset(torch.utils.data.Dataset):
    def __init__(self, encodings, labels):
        self.encodings = encodings
        self.labels = labels

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
        item["labels"] = torch.tensor(self.labels[idx])
        return item

# Create Dataset instances
train_dataset = FakeNewsDataset(train_encodings, train_labels)
val_dataset = FakeNewsDataset(val_encodings, val_labels)
test_dataset = FakeNewsDataset(test_encodings, test_labels)

In [ ]:
# Import the necessary modules
from torch.utils.data import DataLoader # import DataLoader

# Create DataLoaders
batch_size = 16
train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)
test_loader = DataLoader(test_dataset, batch_size=batch_size, shuffle=False)

In [ ]:
from transformers import AutoModelForSequenceClassification, AutoTokenizer

# Load ModernBERT model
model_name = "answerdotai/ModernBERT-base"
model = AutoModelForSequenceClassification.from_pretrained(model_name, num_labels=4)


Some weights of ModernBertForSequenceClassification were not initialized from the model checkpoint at answerdotai/ModernBERT-base and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [ ]:
# Modify the existing `compute_metrics` function to get the per-class metrics

from sklearn.metrics import classification_report

def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return classification_report(labels, preds, output_dict=True)

## 4.5 Hyperparameter Tuning & Training
- Batch size: 16 (FP16 mixed precision)  
- Learning rate: 3e-5 (ModernBERT), 1e-4 (classifier head)  
- Early stopping: Patience=3 on validation loss  
- Gradient accumulation: 4 steps  
*Key Choice*: Lower batch size avoids VRAM issues while maintaining gradient stability.

In [ ]:
from transformers import TrainingArguments, SchedulerType

# Set training arguments
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/ColabNotebooks/W266/FinalProject/model_output",
    evaluation_strategy="epoch",
    save_strategy="epoch",
    learning_rate=3e-5,
    per_device_train_batch_size=16,  # Use smaller batch size
    per_device_eval_batch_size=8,  # Use smaller batch size for evaluation
    num_train_epochs=5,
    weight_decay=0.1,
    logging_dir="/content/drive/MyDrive/ColabNotebooks/W266/FinalProject/logs",
    logging_steps=200,
    load_best_model_at_end=True,
    metric_for_best_model="accuracy",
    report_to="none",
    fp16=True,
    gradient_accumulation_steps=4,  # Accumulate gradients over smaller batches
    lr_scheduler_type=SchedulerType.LINEAR,  # Changed to a valid SchedulerType,
    warmup_steps=0,
)


/usr/local/lib/python3.11/dist-packages/transformers/training_args.py:1611: FutureWarning: `evaluation_strategy` is deprecated and will be removed in version 4.46 of 🤗 Transformers. Use `eval_strategy` instead
  warnings.warn(


In [ ]:
# Import the necessary modules
from torch.utils.data import DataLoader # import DataLoader
from transformers import Trainer, EarlyStoppingCallback # Import the Trainer class
import numpy as np # Import numpy for numerical operations
from sklearn.metrics import accuracy_score, precision_recall_fscore_support # Import accuracy_score and precision_recall_fscore_support

# Initialize Trainer
trainer = Trainer(
    model=model,
    args=training_args,
    train_dataset=train_dataset,
    eval_dataset=val_dataset,
    compute_metrics=compute_metrics,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

# Train the model
trainer.train()

<ipython-input-24-8a05bd2ae793>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
W0412 14:39:49.751000 7169 torch/_inductor/utils.py:1137] [1/0] Not enough SMs to use max_autotune_gemm mode


Epoch,Training Loss,Validation Loss,0,1,2,3,Accuracy,Macro avg,Weighted avg
1,2.669000,0.457383,"{'precision': 0.668903803131991, 'recall': 0.715311004784689, 'f1-score': 0.6913294797687861, 'support': 418.0}","{'precision': 0.7224770642201835, 'recall': 0.7535885167464115, 'f1-score': 0.7377049180327869, 'support': 418.0}","{'precision': 0.9973544973544973, 'recall': 0.9019138755980861, 'f1-score': 0.9472361809045227, 'support': 418.0}","{'precision': 0.8637469586374696, 'recall': 0.8492822966507177, 'f1-score': 0.856453558504222, 'support': 418.0}",0.805024,"{'precision': 0.8131205808360354, 'recall': 0.8050239234449761, 'f1-score': 0.8081810343025795, 'support': 1672.0}","{'precision': 0.8131205808360354, 'recall': 0.8050239234449761, 'f1-score': 0.8081810343025794, 'support': 1672.0}"
2,1.377000,0.425406,"{'precision': 0.6948051948051948, 'recall': 0.7679425837320574, 'f1-score': 0.7295454545454545, 'support': 418.0}","{'precision': 0.7682926829268293, 'recall': 0.7535885167464115, 'f1-score': 0.7608695652173914, 'support': 418.0}","{'precision': 0.9623529411764706, 'recall': 0.9784688995215312, 'f1-score': 0.970344009489917, 'support': 418.0}","{'precision': 0.928, 'recall': 0.8325358851674641, 'f1-score': 0.8776796973518285, 'support': 418.0}",0.833134,"{'precision': 0.8383627047271236, 'recall': 0.8331339712918661, 'f1-score': 0.8346096816511478, 'support': 1672.0}","{'precision': 0.8383627047271237, 'recall': 0.8331339712918661, 'f1-score': 0.8346096816511479, 'support': 1672.0}"
3,0.767900,0.591845,"{'precision': 0.6716417910447762, 'recall': 0.861244019138756, 'f1-score': 0.7547169811320755, 'support': 418.0}","{'precision': 0.8323863636363636, 'recall': 0.7009569377990431, 'f1-score': 0.7610389610389611, 'support': 418.0}","{'precision': 0.9807692307692307, 'recall': 0.9760765550239234, 'f1-score': 0.9784172661870504, 'support': 418.0}","{'precision': 0.9429347826086957, 'recall': 0.8301435406698564, 'f1-score': 0.8829516539440203, 'support': 418.0}",0.842105,"{'precision': 0.8569330420147666, 'recall': 0.8421052631578948, 'f1-score': 0.8442812155755268, 'support': 1672.0}","{'precision': 0.8569330420147666, 'recall': 0.8421052631578947, 'f1-score': 0.8442812155755268, 'support': 1672.0}"
4,0.184300,0.771122,"{'precision': 0.702819956616052, 'recall': 0.7751196172248804, 'f1-score': 0.7372013651877133, 'support': 418.0}","{'precision': 0.8071979434447301, 'recall': 0.7511961722488039, 'f1-score': 0.7781908302354399, 'support': 418.0}","{'precision': 0.9877750611246944, 'recall': 0.9665071770334929, 'f1-score': 0.9770253929866989, 'support': 418.0}","{'precision': 0.8716707021791767, 'recall': 0.861244019138756, 'f1-score': 0.8664259927797834, 'support': 418.0}",0.838517,"{'precision': 0.8423659158411634, 'recall': 0.8385167464114833, 'f1-score': 0.8397108952974088, 'support': 1672.0}","{'precision': 0.8423659158411634, 'recall': 0.8385167464114832, 'f1-score': 0.8397108952974088, 'support': 1672.0}"


<ipython-input-24-8a05bd2ae793>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-24-8a05bd2ae793>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-24-8a05bd2ae793>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}
<ipython-input-24-8a05bd2ae793>:11: UserWarning: To copy construct fro

TrainOutput(global_step=1565, training_loss=0.9872576485045802, metrics={'train_runtime': 3966.0484, 'train_samples_per_second': 25.295, 'train_steps_per_second': 0.395, 'total_flos': 3.461977337554944e+16, 'train_loss': 0.9872576485045802, 'epoch': 4.985645933014354})

## 4.6 Misclassification Analysis  
- Exports 412 misclassified examples to `misclassified_human.csv`  
- Common patterns:  
  - Satire/articles with formal structure  
  - Human texts with low lexical diversity

In [ ]:
# To address the poor performance of human-generated data - Save misclassified examples

val_preds = trainer.predict(val_dataset)
val_df['pred'] = val_preds.predictions.argmax(-1)
misclassified = val_df[val_df['label'].map(label_map) != val_df['pred']]
misclassified[['text', 'label', 'pred']].to_csv('misclassified_human.csv', index=False)

<ipython-input-24-8a05bd2ae793>:11: UserWarning: To copy construct from a tensor, it is recommended to use sourceTensor.clone().detach() or sourceTensor.clone().detach().requires_grad_(True), rather than torch.tensor(sourceTensor).
  item = {key: torch.tensor(val[idx]) for key, val in self.encodings.items()}


# 5. Deployment & Extensions

## 5.1 Model Packaging  
- Saves full model + tokenizer to Google Drive  
- Includes configuration.json and training logs  
*Reproducibility*: Full export allows reloading without code dependencies.

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM

ModernBERT_model_checkpoint_filepath = 'drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/ModernBERT_model'

# Save
model.save_pretrained(ModernBERT_model_checkpoint_filepath)
tokenizer.save_pretrained(ModernBERT_model_checkpoint_filepath)

('drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/ModernBERT_model/tokenizer_config.json',
 'drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/ModernBERT_model/special_tokens_map.json',
 'drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/ModernBERT_model/tokenizer.json')

## 5.2 Prediction Demo

In [ ]:
from transformers import BertTokenizer, BertForMaskedLM, AutoTokenizer

# Path to your saved model and tokenizer
ModernBERT_model_checkpoint_filepath = 'drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/ModernBERT_model'

# Load the model and tokenizer from the saved directory
model = BertForMaskedLM.from_pretrained(ModernBERT_model_checkpoint_filepath)

# Use AutoTokenizer to automatically load the correct tokenizer based on the model
tokenizer = AutoTokenizer.from_pretrained(ModernBERT_model_checkpoint_filepath)

#Test the loaded model
inputs = tokenizer("Celebrities vow to leave U.S. if Trump wins 2024 election", return_tensors='pt')
outputs = model(**inputs)
print(tokenizer.decode(outputs.logits.argmax(-1)[0]))

You are using a model of type modernbert to instantiate a model of type bert. This is not supported for all configurations of models and can yield errors.
Some weights of BertForMaskedLM were not initialized from the model checkpoint at drive/MyDrive/Colab Notebooks/W266/FinalProject/model_checkpoints/ModernBERT_model and are newly initialized: ['bert.embeddings.LayerNorm.bias', 'bert.embeddings.LayerNorm.weight', 'bert.embeddings.position_embeddings.weight', 'bert.embeddings.token_type_embeddings.weight', 'bert.embeddings.word_embeddings.weight', 'bert.encoder.layer.0.attention.output.LayerNorm.bias', 'bert.encoder.layer.0.attention.output.LayerNorm.weight', 'bert.encoder.layer.0.attention.output.dense.bias', 'bert.encoder.layer.0.attention.output.dense.weight', 'bert.encoder.layer.0.attention.self.key.bias', 'bert.encoder.layer.0.attention.self.key.weight', 'bert.encoder.layer.0.attention.self.query.bias', 'bert.encoder.layer.0.attention.self.query.weight', 'bert.encoder.layer.0.atte

 Exception Exception Jazz Exception Exception Jazz157uish ExceptionPORT Exceptionsell Exception ExceptionOb Neural Exception Neural


# 6. Model Experimentation

Implement a hybrid neural network architecture that combines a pre-trained ModernBERT language model with custom linguistic features for text classification

## 6.1 Data Preparation:

1. Tokenization: Uses ModernBERT's tokenizer to convert text into input IDs and attention masks

2. Feature Engineering: Adds custom linguistic features (via add_linguistic_features function) such as:

- Readability scores

- Sentence complexity metrics

- Vocabulary diversity measures

- Syntactic features

In [ ]:
# Import the necessary modules
import torch
import torch.nn as nn
from transformers import AutoModel, AutoTokenizer, AutoConfig
from torch.utils.data import Dataset, DataLoader
import numpy as np
from torch.cuda.amp import autocast, GradScaler

# Pre-tokenize all texts
def pre_tokenize(texts, tokenizer, max_length=512):
    return tokenizer(
        texts.tolist(), # Convert pandas Series to list
        truncation=True,
        padding='max_length',
        max_length=max_length,
        return_tensors='pt'
    )

# train_encodings = pre_tokenize(train_df['cleaned_text'], tokenizer)
# val_encodings = pre_tokenize(val_df['cleaned_text'], tokenizer)

## 6.2 Hybrid Dataset:

Combines three elements for each data point:

- BERT token encodings (input_ids, attention_mask)

- Linguistic features

- Classification labels

In [ ]:
# 2. Hybrid Dataset Class
class HybridDataset(torch.utils.data.Dataset):
    def __init__(self, texts, features, labels, tokenizer, max_length=512):
        self.encodings = tokenizer(
            texts,
            truncation=True,
            padding='max_length',
            max_length=max_length,
            return_tensors='pt'
        )
        self.features = torch.tensor(features, dtype=torch.float32)
        self.labels = torch.tensor(labels, dtype=torch.long)

    def __len__(self):
        return len(self.labels)

    def __getitem__(self, idx):
        return {
            'input_ids': self.encodings['input_ids'][idx],
            'attention_mask': self.encodings['attention_mask'][idx],
            'linguistic_features': self.features[idx],
            'labels': self.labels[idx]
        }

In [ ]:
# 2.1. Update Data Preparation and Loaders (Assuming train_features, val_features are already computed)
train_dataset = HybridDataset(train_df['cleaned_text'].tolist(), train_features, train_labels, tokenizer) # Pass tokenizer here
val_dataset = HybridDataset(val_df['cleaned_text'].tolist(), val_features, val_labels, tokenizer) # Pass tokenizer here

# Create val_features to align with val_dataset
val_features = [add_linguistic_features(text) for text in val_df['cleaned_text']]

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
val_loader = DataLoader(val_dataset, batch_size=batch_size, shuffle=False)

## 6.3 Hybird Model Setup

- Base Model: ModernBERT (pretrained transformer)

- Feature Fusion: Concatenates BERT's CLS token embedding with linguistic features

- Classification Head: Processes combined features through dropout and linear layer

In [ ]:
# 3. Hybrid Model Architecture
from transformers import AutoModel
import torch.nn as nn

class HybridModernBERT(nn.Module):
    def __init__(self, model_name="answerdotai/ModernBERT-base", num_features=10, num_labels=4):
        super().__init__()
        self.bert = AutoModel.from_pretrained(model_name)
        self.dropout = nn.Dropout(0.2)
        self.classifier = nn.Linear(self.bert.config.hidden_size + num_features, num_labels)

    def forward(self, input_ids, attention_mask, linguistic_features):
        bert_output = self.bert(
            input_ids=input_ids,
            attention_mask=attention_mask
        )
        pooled = bert_output.last_hidden_state[:, 0, :]  # CLS token
        combined = torch.cat([pooled, linguistic_features], dim=1)
        return self.classifier(self.dropout(combined))

In [ ]:
# 4. Initialize Components
model = HybridModernBERT()
tokenizer = AutoTokenizer.from_pretrained("answerdotai/ModernBERT-base")

In [ ]:
# 5. Data Preparation
from sklearn.preprocessing import StandardScaler

# Precompute features
train_features = [add_linguistic_features(text) for text in train_df['cleaned_text']]
val_features = [add_linguistic_features(text) for text in val_df['cleaned_text']]

# Label Encoding: the model requires numerical labels
label_map = {
    "human_real": 0,
    "human_fake": 1,
    "gpt_fake": 2,
    "gpt_real": 3  # Added
}

# Ensure val_df is not empty here and check unique labels
print(f"Validation DataFrame size: {len(val_df)}")
print(f"Unique labels in val_df['label']: {val_df['label'].unique()}")

train_labels = train_df['label'].map(label_map).tolist()
val_labels = val_df['label'].map(label_map).tolist()
test_labels = test_df['label'].map(label_map).tolist()

# Create Hybrid Datasets (keep existing train_dataset and val_dataset)
train_dataset_hybrid = HybridDataset(train_df['cleaned_text'].tolist(), train_features, train_labels, tokenizer)
val_dataset_hybrid = HybridDataset(val_df['cleaned_text'].tolist(), val_features, val_labels, tokenizer)

# Create Hybrid DataLoaders
train_loader_hybrid = DataLoader(train_dataset_hybrid, batch_size=batch_size, shuffle=True)
val_loader_hybrid = DataLoader(val_dataset_hybrid, batch_size=batch_size, shuffle=False)

# Check if datasets are empty
if len(train_dataset_hybrid) == 0:
    raise ValueError("Training dataset is empty!")
if len(val_dataset_hybrid) == 0:
    raise ValueError("Validation dataset is empty!")

# Check if dataloaders are empty
if len(train_loader_hybrid) == 0:
    raise ValueError("Training dataloader is empty!")
if len(val_loader_hybrid) == 0:
    raise ValueError("Validation dataloader is empty!")

# Check if val_labels has any data
print(f"Validation labels size: {len(val_labels)}")

# Inspect the first few elements of val_labels (if not empty)
if val_labels:
    print(f"First few validation labels: {val_labels[:10]}")

Validation DataFrame size: 1672
Unique labels in val_df['label']: ['human_real' 'human_fake' 'gpt_fake' 'gpt_real']
Validation labels size: 1672
First few validation labels: [0, 0, 0, 0, 0, 0, 0, 0, 0, 0]


In [ ]:
# 6. Training Setup - Custom training
optimizer = torch.optim.AdamW([
    {'params': model.bert.parameters(), 'lr': 2e-5},
    {'params': model.classifier.parameters(), 'lr': 1e-4}
])

## 6.4 Hybrid Model Training

- 5-epoch training loop

- Batch processing with cross-entropy loss

- Gradient backpropagation and optimization

- Model checkpoint saving

In [ ]:
# 7. Custom Training Loop
from tqdm import tqdm
import time
from torch.nn import functional as F  # Import functional
from sklearn.metrics import f1_score, precision_score, recall_score, accuracy_score, classification_report

# Modified Training Loop with Metrics
scaler = torch.amp.GradScaler()

# Get the device
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.to(device) # Move the model to the appropriate device

# Add class label mapping
class_names = ['human_real', 'human_fake', 'gpt_fake', 'gpt_real']

# Enable mixed precision globally (before any training steps)
torch.backends.cuda.matmul.allow_tf32 = True
torch.backends.cudnn.benchmark = True
torch.backends.cudnn.deterministic = False
torch.backends.cudnn.allow_tf32 = True


for epoch in range(5):
    # Training phase
    model.train()
    all_preds = []
    all_labels = []

    for batch in train_loader_hybrid:
        optimizer.zero_grad()

        input_ids = batch['input_ids'].to(device)
        attention_mask = batch['attention_mask'].to(device)
        linguistic_features = batch['linguistic_features'].to(device)
        labels = batch['labels'].to(device)

        # Fix mixed precision context
        with torch.amp.autocast(device_type=device.type, dtype=torch.float16):
            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                linguistic_features=linguistic_features
            )
            loss = F.cross_entropy(outputs, labels)
        # Gradient clipping to prevent exploding gradients
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)  # Clip gradients

        loss.backward() # Calculate the gradients directly

        if not torch.isfinite(loss): # Check if loss is finite using torch.isfinite
            print("Encountered inf/NaN in loss. Skipping this batch.")
            continue
        optimizer.step()

        # Store predictions
        all_preds.extend(torch.argmax(outputs.detach(), 1).cpu().numpy())
        all_labels.extend(labels.cpu().numpy())

    # Training report with class names
    print(f"\nEpoch {epoch+1} Training Report:")
    print(classification_report(all_labels, all_preds, target_names=class_names, digits=4))

    # Validation phase
    model.eval()
    val_preds = []
    val_labels = []

    with torch.no_grad():
        for batch in val_loader_hybrid:
            input_ids = batch['input_ids'].to(device)
            attention_mask = batch['attention_mask'].to(device)
            linguistic_features = batch['linguistic_features'].to(device)
            labels = batch['labels'].to(device)

            outputs = model(
                input_ids=input_ids,
                attention_mask=attention_mask,
                linguistic_features=linguistic_features
            )

            val_preds.extend(torch.argmax(outputs, 1).cpu().numpy())
            val_labels.extend(labels.cpu().numpy())

    # Validation report with class names
    print(f"Epoch {epoch+1} Validation Report:")
    print(classification_report(val_labels, val_preds, target_names=class_names, digits=4))


Epoch 1 Training Report:
              precision    recall  f1-score   support

  human_real     0.5417    0.5510    0.5463      6688
  human_fake     0.5655    0.5694    0.5674      6688
    gpt_fake     0.7595    0.7557    0.7576      3344
    gpt_real     0.6112    0.5849    0.5978      3344

    accuracy                         0.5969     20064
   macro avg     0.6195    0.6152    0.6173     20064
weighted avg     0.5975    0.5969    0.5971     20064

Epoch 1 Validation Report:
              precision    recall  f1-score   support

  human_real     0.6485    0.6842    0.6659       418
  human_fake     0.6660    0.8158    0.7333       418
    gpt_fake     0.9633    0.9426    0.9528       418
    gpt_real     0.9710    0.7201    0.8269       418

    accuracy                         0.7907      1672
   macro avg     0.8122    0.7907    0.7947      1672
weighted avg     0.8122    0.7907    0.7947      1672


Epoch 2 Training Report:
              precision    recall  f1-score   suppo

In [ ]:
# 10. Save Model
torch.save(model.state_dict(), "hybrid_model.pth")

In [ ]:
# Recreate model architecture
loaded_model = HybridModernBERT()
loaded_model.load_state_dict(torch.load("hybrid_model.pth"))
loaded_model.to(device)

# Example inference
loaded_model.eval()
with torch.no_grad():
    input_ids = tokenizer("Your text here", return_tensors="pt").to(device)
    features = torch.tensor([add_linguistic_features("Your text here")], dtype=torch.float32).to(device)
    outputs = loaded_model(input_ids['input_ids'], input_ids['attention_mask'], features)
    predictions = torch.argmax(outputs, dim=1)